In [1]:
!pip install -q langgraph langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.6 MB/s eta 0:00:00


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key="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",
    model_name="llama-3.1-8b-instant",
    temperature=0
)

In [3]:
from typing import TypedDict

class TeamState(TypedDict):
    task: str
    worker_result: str
    summary: str

In [4]:
def worker(state: TeamState):
    answer = llm.invoke(
        f"Solve this math problem. Return only the final number.\n\n{state['task']}"
    ).content

    return {"worker_result": answer}


def supervisor(state: TeamState):
    summary = llm.invoke(
        f"""
The worker solved:
Question: {state['task']}
Answer: {state['worker_result']}

Write a one-line summary.
"""
    ).content

    return {"summary": summary}

In [5]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(TeamState)

builder.add_node("worker", worker)
builder.add_node("supervisor", supervisor)

builder.add_edge(START, "worker")
builder.add_edge("worker", "supervisor")
builder.add_edge("supervisor", END)

graph = builder.compile()

In [6]:
result = graph.invoke(
    {
        "task": "What is 144 divided by 12, then plus 5?"
    }
)

print("Worker result:", result["worker_result"])
print("Supervisor summary:", result["summary"])

Worker result: 12
Supervisor summary: The worker solved a math problem by dividing 144 by 12, which equals 12, and then adding 5, resulting in a final answer of 17.
